# 환경 준비 (setup)

`run.ipynb` 를 돌리기 전에 **한 번만** 실행합니다. 위에서 아래로.

```
1 점검 → 2 파이썬 → 3 패키지 → 4 .env → 5 ffmpeg
      → 6 회귀테스트 → 7 스모크 → 8 드라이브 → 9 최종체크
```

6번까지 통과하면 `run.ipynb` 로 넘어갈 수 있습니다. 7~8번은 선택입니다.

## 1. 현재 상태 점검

무엇이 없는지 먼저 봅니다. 아무것도 바꾸지 않습니다.

In [1]:
import importlib.util, os, shutil, subprocess, sys
from pathlib import Path

OK, NO = chr(79), chr(88)          # O / X
ROOT = Path.cwd()
if not (ROOT / 'src').is_dir():
    raise SystemExit(f'meeting_minutes 폴더에서 열어야 합니다. 현재: {ROOT}')
sys.path.insert(0, str(ROOT))

PKGS = [
    ('anthropic', 'Claude API'),
    ('pydantic', '스키마'),
    ('jinja2', '렌더'),
    ('faster_whisper', '로컬 STT'),
    ('googleapiclient', '구글 드라이브'),
    ('dotenv', '.env 로드'),
]

def have(mod):
    try:
        return importlib.util.find_spec(mod) is not None
    except (ImportError, ValueError):
        return False

print(f'python   {sys.version.split()[0]}')
print(f'kernel   {sys.executable}')
print(f'ffmpeg   {shutil.which('ffmpeg') or NO}')
print()
for mod, desc in PKGS:
    print(f'  {OK if have(mod) else NO}  {mod:18} {desc}')
print()
for f in ['.env', 'credentials.json', 'token.json']:
    print(f'  {OK if (ROOT / f).exists() else NO}  {f}')
print(f'  {OK if os.getenv('ANTHROPIC_API_KEY') else NO}  ANTHROPIC_API_KEY (환경변수)')

python   3.14.0
kernel   c:\Users\skswl\Desktop\Github\v02_quiz_builder\.venv\Scripts\python.exe
ffmpeg   X

  X  anthropic          Claude API
  O  pydantic           스키마
  X  jinja2             렌더
  X  faster_whisper     로컬 STT
  X  googleapiclient    구글 드라이브
  X  dotenv             .env 로드

  X  .env
  X  credentials.json
  X  token.json
  X  ANTHROPIC_API_KEY (환경변수)


## 2. 파이썬 버전

**3.12 권장.** 3.13/3.14 는 `faster-whisper` 가 쓰는 CTranslate2 휠이 아직 없을 수 있습니다.

3.12 커널을 만들려면 터미널에서:

```powershell
py -3.12 -m venv .venv
.\.venv\Scripts\Activate.ps1
pip install ipykernel
python -m ipykernel install --user --name mm312 --display-name "Python 3.12 (meeting_minutes)"
```

그 다음 Jupyter 우상단에서 커널을 바꾸고 1번부터 다시 실행하세요.

In [2]:
major, minor = sys.version_info[:2]
if (major, minor) == (3, 12):
    print('OK  파이썬 3.12 - 그대로 진행하세요')
elif (major, minor) >= (3, 13):
    print(f'주의  {major}.{minor} 입니다. faster-whisper 설치가 실패할 수 있습니다.')
    print('      STT 없이 (녹취록 텍스트로만) 쓸 거면 그대로 진행해도 됩니다.')
else:
    print(f'주의  {major}.{minor} 는 너무 낮습니다. 3.12 를 쓰세요.')

주의  3.14 입니다. faster-whisper 설치가 실패할 수 있습니다.
      STT 없이 (녹취록 텍스트로만) 쓸 거면 그대로 진행해도 됩니다.


## 3. 패키지 설치

`requirements.txt` 를 현재 커널에 설치합니다. 몇 분 걸립니다.

STT 를 안 쓸 거면 `SKIP_STT = True` — `faster-whisper` 가 설치 중 가장 무겁습니다.

In [3]:
SKIP_STT = False      # True 면 faster-whisper 제외

req = ROOT / 'requirements.txt'
lines = [l.strip() for l in req.read_text(encoding='utf-8').splitlines()]
pkgs = [l for l in lines if l and not l.startswith('#')]
if SKIP_STT:
    pkgs = [p for p in pkgs if 'whisper' not in p]

print('설치 대상:', ', '.join(pkgs))
print()
# 자식 프로세스 출력은 UTF-8 로 받는다. text=True 만 쓰면 Windows 로케일(cp949)로
# 디코딩하다 한글에서 터지고 r.stdout 이 None 이 된다.
CHILD_ENV = {**os.environ, 'PYTHONIOENCODING': 'utf-8'}
r = subprocess.run([sys.executable, '-m', 'pip', 'install', *pkgs],
                   capture_output=True, text=True,
                   encoding='utf-8', errors='replace', env=CHILD_ENV)
for l in ((r.stdout or '') + (r.stderr or '')).strip().splitlines()[-12:]:
    print(l)
print()
print('성공' if r.returncode == 0 else f'실패 (exit {r.returncode}) - 위 로그 확인')

설치 대상: anthropic, pydantic, jinja2, faster-whisper, google-api-python-client, google-auth-oauthlib, python-dotenv, jupyterlab, ipykernel

   ---------------------------------------  74/75 [jupyterlab]
   ---------------------------------------  74/75 [jupyterlab]
   ---------------------------------------  74/75 [jupyterlab]
   ---------------------------------------  74/75 [jupyterlab]
   ---------------------------------------  74/75 [jupyterlab]
   ---------------------------------------  74/75 [jupyterlab]
   ---------------------------------------- 75/75 [jupyterlab]


[notice] A new release of pip is available: 25.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip

성공


## 4. `.env`

`.env.example` 을 복사합니다. **이미 있으면 건드리지 않습니다.**

만든 뒤 `ANTHROPIC_API_KEY=` 뒤에 키를 넣어야 추출이 됩니다. `.env` 는 gitignore 돼 있습니다.

In [4]:
env, example = ROOT / '.env', ROOT / '.env.example'
if env.exists():
    print('.env 가 이미 있습니다 - 그대로 둡니다')
else:
    env.write_text(example.read_text(encoding='utf-8'), encoding='utf-8')
    print(f'생성: {env}')
print()
for line in env.read_text(encoding='utf-8').splitlines():
    if line.startswith('ANTHROPIC_API_KEY'):
        filled = len(line.split('=', 1)[1].strip()) > 0
        print('OK  키가 채워져 있습니다' if filled
              else '할 일  .env 를 열어 ANTHROPIC_API_KEY= 뒤에 키를 넣으세요')
        break

생성: c:\Users\skswl\Desktop\Github\AI_hub\office_automation\meeting_minutes\.env

할 일  .env 를 열어 ANTHROPIC_API_KEY= 뒤에 키를 넣으세요


## 5. ffmpeg

whisper 가 오디오를 디코딩할 때 필요합니다. **녹취록 텍스트만 쓸 거면 없어도 됩니다.**

노트북에서 설치하지 않습니다 (권한 창이 뜰 수 있어서). 터미널에서:

```powershell
winget install Gyan.FFmpeg
```

설치 후 **Jupyter 를 다시 시작**해야 PATH 가 반영됩니다.

In [6]:
found = shutil.which('ffmpeg')
if found:
    print(f'OK  {found}')
else:
    print('없음 - 오디오 파일부터 시작하려면 필요합니다')
    print('     녹취록 텍스트로만 쓸 거면 건너뛰어도 됩니다')

없음 - 오디오 파일부터 시작하려면 필요합니다
     녹취록 텍스트로만 쓸 거면 건너뛰어도 됩니다


## 6. 회귀 테스트 (관문)

API 도 오디오도 필요 없는 테스트 66개입니다. **여기가 통과하면 코드는 정상입니다.**

실패하면 `run.ipynb` 로 넘어가지 마세요.

In [7]:
# 출력을 UTF-8 로 받는다 (cp949 로 디코딩하면 한글에서 터진다)
CHILD_ENV = {**os.environ, 'PYTHONIOENCODING': 'utf-8'}
r = subprocess.run([sys.executable, str(ROOT / 'tests' / 'test_offline.py')],
                   capture_output=True, text=True, cwd=str(ROOT),
                   encoding='utf-8', errors='replace', env=CHILD_ENV)
out = ((r.stdout or '') + (r.stderr or '')).splitlines()
fails = [l for l in out if '[FAIL]' in l]
for l in out[-6:]:
    print(l)
if fails:
    print()
    print('실패 목록:')
    for l in fails:
        print('  ', l.strip())
print()
print('통과' if r.returncode == 0 else '실패 - run.ipynb 로 넘어가지 마세요')

  [PASS] 한글 유지
  [PASS] 길이 제한

  PASS 66  /  FAIL 0

통과


## 7. 스모크 — API 없이 산출물까지

가짜 회의록으로 md/html/json 이 실제로 만들어지는지 봅니다. **키가 없어도 됩니다.**
임시 폴더에 만들고 바로 지웁니다.

In [8]:
import tempfile
from src.schema import Minutes, MinutesBundle, ActionItem, Decision, Topic
from src.render import render
from src.review import render_review

m = Minutes(
    title='스모크 테스트', date='2026-08-25', one_liner='산출물이 만들어지는지 확인',
    topics=[Topic(title='확인', summary='렌더 경로 점검')],
    decisions=[Decision(decision='파이프라인 정상', quote='그렇게 하죠')],
    action_items=[
        ActionItem(task='담당 명시된 일', owner='김PM', due='2026-09-01', quote='q1'),
        ActionItem(task='회의에서 안 정해진 일', owner_status='not_stated',
                   due_status='not_stated', quote='q2'),
        ActionItem(task='녹취가 불확실한 일', owner_status='unclear',
                   due_status='unclear', quote='q3'),
    ],
)
b = MinutesBundle(minutes=m, model='(스모크)', generated_at='2026-08-25 00:00')

tmp = Path(tempfile.mkdtemp(prefix='mm_smoke_'))
try:
    out = render(b, tmp)
    md_text = out.md.read_text(encoding='utf-8')
    print('생성:', out.md.name, '/', out.html.name, '/', out.json.name)
    print()
    print('빈칸 이유가 문서까지 갔는가:')
    for label in ['회의에서 안 정해짐', '녹취 불확실']:
        print(f'  {OK if label in md_text else NO}  {label}')
    print()
    print(render_review(m))
finally:
    shutil.rmtree(tmp, ignore_errors=True)

[render] 2026-08-25_스모크_테스트.md / 2026-08-25_스모크_테스트.html / 2026-08-25_스모크_테스트.json
생성: 2026-08-25_스모크_테스트.md / 2026-08-25_스모크_테스트.html / 2026-08-25_스모크_테스트.json

빈칸 이유가 문서까지 갔는가:
  O  회의에서 안 정해짐
  O  녹취 불확실

  스모크 테스트   2026-08-25
  산출물이 만들어지는지 확인

[결정사항]
  D1. 파이프라인 정상
      근거: "그렇게 하죠"

[액션아이템]
  A1. 담당 명시된 일
      담당 김PM / 마감 2026-09-01 / unknown
  A2. 회의에서 안 정해진 일
      담당 <회의에서 안 정해짐> / 마감 <회의에서 안 정해짐> / unknown
  A3. 녹취가 불확실한 일
      담당 <녹취 불확실 · 오디오 재확인> / 마감 <녹취 불확실 · 오디오 재확인> / unknown

------------------------------------------------------------------------
반영할 항목을 고르세요. 회의에서 나온 말이 전부 결정은 아닙니다.
  python -m src.pipeline --accept D1,D2,A1,A3  --draft <draft.json>
  python -m src.pipeline --accept all          --draft <draft.json>
------------------------------------------------------------------------


## 8. 구글 드라이브 (선택)

업로드를 쓸 때만 필요합니다. 이 노트북에서 인증하지 않습니다 — 첫 업로드 때 브라우저가 열립니다.

1. Google Cloud Console → 프로젝트 → **Google Drive API 사용 설정**
2. 사용자 인증 정보 → OAuth 클라이언트 ID → **데스크톱 앱** → JSON 다운로드
3. 이 폴더에 `credentials.json` 으로 저장

In [9]:
cred = ROOT / 'credentials.json'
if cred.exists():
    print('OK  credentials.json 있음 - 첫 업로드 때 브라우저 인증이 한 번 열립니다')
else:
    print('없음 - 업로드 없이 로컬 저장만 가능 (run.ipynb 의 UPLOAD=False)')

없음 - 업로드 없이 로컬 저장만 가능 (run.ipynb 의 UPLOAD=False)


## 9. 최종 체크

In [10]:
rows = [
    ('파이썬 3.12', sys.version_info[:2] == (3, 12), 'STT 쓰려면 필수'),
    ('anthropic', have('anthropic'), '추출에 필수'),
    ('jinja2', have('jinja2'), '렌더에 필수'),
    ('.env', (ROOT / '.env').exists(), '필수'),
    ('API 키', bool(os.getenv('ANTHROPIC_API_KEY')), '.env 채우고 커널 재시작'),
    ('faster-whisper', have('faster_whisper'), '오디오부터 시작할 때만'),
    ('ffmpeg', bool(shutil.which('ffmpeg')), '오디오부터 시작할 때만'),
    ('credentials.json', (ROOT / 'credentials.json').exists(), '드라이브 업로드할 때만'),
]

for name, okv, note in rows:
    print(f'  {OK if okv else NO}  {name:18} {note}')
print()
must = all(r[1] for r in rows[1:5])
if must:
    print('준비 완료 - run.ipynb 를 여세요')
else:
    print('필수 항목이 비어 있습니다. 위 X 를 먼저 채우세요')
    print('(.env 를 방금 채웠다면 커널 재시작이 필요합니다)')

  X  파이썬 3.12           STT 쓰려면 필수
  O  anthropic          추출에 필수
  O  jinja2             렌더에 필수
  O  .env               필수
  X  API 키              .env 채우고 커널 재시작
  O  faster-whisper     오디오부터 시작할 때만
  X  ffmpeg             오디오부터 시작할 때만
  X  credentials.json   드라이브 업로드할 때만

필수 항목이 비어 있습니다. 위 X 를 먼저 채우세요
(.env 를 방금 채웠다면 커널 재시작이 필요합니다)


---

## 다음

| 상황 | 할 것 |
|---|---|
| 6번 통과 + API 키 있음 | **`run.ipynb`** 로 이동 |
| STT 없이 쓰고 싶음 | 녹취록을 `data/transcripts/` 에 두고 `run.ipynb` 의 `TRANSCRIPT` 지정 |
| 6번 실패 | 실패 목록 확인. `run.ipynb` 로 넘어가지 말 것 |
| .env 를 방금 채웠음 | 커널 재시작 후 9번 다시 실행 |